# Preguntas y tasas de medalla (Puntos 9 y 10)
Se cargan los datos igual que en el notebook 01.

In [ ]:
import pandas as pd
import numpy as np

df_atletas = pd.read_csv('../data/athlete_events.csv')
df_regiones = pd.read_csv('../data/noc_regions.csv')

df_atletas['IMC'] = df_atletas['Weight'] / ((df_atletas['Height'] / 100) ** 2)
df_atletas = pd.merge(df_atletas, df_regiones, on='NOC', how='left')
df_atletas['medalla'] = df_atletas['Medal'].notna().astype(int)

print(df_atletas.shape)

### 9. Preguntas puntuales

In [ ]:
print("a) Edad promedio:", round(df_atletas['Age'].mean(), 2))

print("\nb) Registros con y sin medalla:")
print(df_atletas['medalla'].value_counts().rename({0: 'Sin medalla', 1: 'Con medalla'}))

print("\nc) Estatura promedio de los que ganaron oro:",
      round(df_atletas.loc[df_atletas['Medal'] == 'Gold', 'Height'].mean(), 2))

print("\nd) Ediciones distintas:", df_atletas['Games'].nunique())
print("   Países (NOC) distintos:", df_atletas['NOC'].nunique())

print("\ne) Edad mínima:", df_atletas['Age'].min(), " Edad máxima:", df_atletas['Age'].max())

print("\nf) Registros por temporada:")
print(df_atletas['Season'].value_counts())

ediciones = df_atletas.groupby('ID')['Games'].nunique()
print("\ng) Atletas en una sola edición:", (ediciones == 1).sum())
print("   Atletas en más de una edición:", (ediciones > 1).sum())

### Respuestas

* **a)** La edad promedio es de unos 25.6 años.
* **b)** 39,783 registros con medalla y 231,333 sin medalla, o sea que solo cerca del 15% de las participaciones terminan en medalla.
* **c)** El resultado aparece en la salida. Se calcula solo con los que tienen estatura registrada.
* **d)** Hay 51 ediciones y 230 códigos NOC distintos. Ojo que un mismo país puede tener varios códigos en la historia.
* **e)** La edad mínima es 10 años y la máxima 97. La de 97 llama la atención y seguramente corresponde a competencias de arte, que antes eran parte de los Juegos.
* **f)** 222,552 registros de Verano y 48,564 de Invierno.
* **g)** Los valores aparecen en la salida. La mayoría de atletas solo participa en una edición.

### 10. Hipótesis y verificación con tasas

In [ ]:
def resumen(col):
    t = df_atletas.groupby(col)['medalla'].agg(medallas='sum', registros='count')
    t['tasa_%'] = (t['medallas'] / t['registros'] * 100).round(2)
    return t

# a) mujeres vs hombres
display(resumen('Sex'))

# b) menores de 18 vs adultos (sin contar los que no tienen edad)
edad = df_atletas.dropna(subset=['Age']).copy()
edad['menor_18'] = np.where(edad['Age'] < 18, 'Menor de 18', '18 o más')
t_b = edad.groupby('menor_18')['medalla'].agg(medallas='sum', registros='count')
t_b['tasa_%'] = (t_b['medallas'] / t_b['registros'] * 100).round(2)
display(t_b)

# c) menores de 18 o 35+ vs el resto
edad['grupo'] = np.where((edad['Age'] < 18) | (edad['Age'] >= 35), 'Menor de 18 o 35+', '18 a 34')
t_c = edad.groupby('grupo')['medalla'].agg(medallas='sum', registros='count')
t_c['tasa_%'] = (t_c['medallas'] / t_c['registros'] * 100).round(2)
display(t_c)

# d) USA vs otros paises
df_atletas['es_usa'] = np.where(df_atletas['NOC'] == 'USA', 'USA', 'Otros países')
display(resumen('es_usa'))

# e) verano vs invierno
display(resumen('Season'))

resumen('Sex').to_csv('../outputs/tablas/medallas_por_sexo.csv')
resumen('Season').to_csv('../outputs/tablas/medallas_por_temporada.csv')

### Hipótesis y resultados

* **a) Mujeres vs hombres.** Hipótesis: los hombres ganaron más medallas porque hay muchos más hombres en los datos, pero proporcionalmente la diferencia debería ser pequeña. Con la tabla se confirma que en conteo los hombres ganan muchas más medallas, y al comparar la tasa se ve que la diferencia es mucho menor, porque las mujeres también tienen menos registros.
* **b) Menores de 18 vs adultos.** Hipótesis: los adultos ganan más medallas en conteo y también en proporción, porque tienen más experiencia y desarrollo físico. La tabla lo confirma: los menores son pocos registros y su tasa de medalla es más baja.
* **c) Menores de 18 o 35+ vs el resto.** Hipótesis: el grupo de 18 a 34 gana más porque es la edad de mejor rendimiento. En conteo gana claramente el grupo de 18 a 34, y en proporción también tiene la tasa más alta, aunque los mayores de 35 tienen una tasa nada despreciable en deportes como tiro o equitación.
* **d) USA vs otros países.** Hipótesis: en conteo ganan los otros países porque son muchos más, pero proporcionalmente USA tiene una tasa mucho más alta. La tabla lo confirma: USA es un solo país con una tasa de medalla muy superior al promedio.
* **e) Verano vs Invierno.** Hipótesis: en Verano se ganan más medallas porque hay muchas más pruebas. En conteo gana Verano por mucho, y en tasa Verano también queda un poco arriba, aunque la diferencia es menor que en el conteo.
* **Conclusión:** comparar solo conteos puede engañar, porque los grupos tienen tamaños muy distintos. Las tasas permiten ver mejor quién tiene más probabilidad de ganar.